# ISLR Pipeline — LIBRAS Landmark Subset
Pipeline completo: extração → filtro → treino → resultados

## 0. Setup

In [1]:
import os, sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
os.chdir(PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

print('Raiz do projeto:', PROJECT_ROOT)

Raiz do projeto: /Users/dani/Desktop/1


In [2]:
%pip install -r requirements.txt
# %pip install --upgrade torch torchvision

# PYTORCH_ENABLE_MPS_FALLBACK=1 python scripts/03_train.py ...


Note: you may need to restart the kernel to use updated packages.


In [23]:
# ── Configuração central — edite aqui ─────────────────────────────────────

DATASET      = 'ufop'          # 'minds' | 'ufop' | 'ksl' | 'include50'
SUBSET       = 'all'          # 'all' | '1st' | '2nd' | 'laines' | 'arcanjo'
EXTRACTOR    = 'mediapipe'    # 'mediapipe' | 'openpose'
MODEL        = 'resnet18'     # 'resnet18' | 'resnet50' | 'efficientnet_b6' | 'vit_medium'
IMAGE_METHOD = 'Skeleton-DML' # 'Skeleton-DML' | 'Skeleton-Magnitude' | 'SL-DML'
IMPUTE       = True
REF          = 'ksl_all_com_imputacao' # referência para salvar os resultados (nome do experimento)

LEARNING_RATE = 0.0001
WEIGHT_DECAY  = 0.0001
EPOCHS        = 30 #50
SEED          = 42
PATIENCE      = 5

RAW_DIR       = 'data/raw'
INTERIM_DIR   = 'data/interim'
PROCESSED_DIR = 'data/processed'
OUTPUT_DIR    = 'experiments'
TABLE_DIR     = 'reports/tables'

# Datasets que usam CSV direto (sem extração de vídeo)
CSV_DATASETS = {'ksl', 'include50'}

print('Config ok')

Config ok


---
## 1. Extração de landmarks
Pule se já tiver o CSV em `data/interim/<dataset>/<dataset>_mediapipe.csv`.

In [24]:
from datasets.base_dataset import BaseVideoDataset
from extraction.base_extractor import BaseExtractor

interim_csv = os.path.join(INTERIM_DIR, DATASET, f'{DATASET}_{EXTRACTOR}.csv')

if os.path.exists(interim_csv):
    print(f'CSV já existe: {interim_csv}\nDelete o arquivo se quiser re-extrair.')
else:
    dataset_instance = BaseVideoDataset.create(DATASET, RAW_DIR)
    extractor        = BaseExtractor.create(EXTRACTOR)
    processor        = dataset_instance.get_processor(extractor)
    videos           = dataset_instance.prepare_data()

    print(f'{len(videos)} vídeos encontrados')
    os.makedirs(os.path.join(INTERIM_DIR, DATASET), exist_ok=True)
    processor.process_all(videos, interim_csv, chunk_size=10_000)
    print(f'Extração concluída → {interim_csv}')

ValueError: Labels file not found: data/raw/ufop/labels.txt

---
## 2. Filtro + imputação
Pule se já tiver o CSV em `data/processed/<dataset>/<dataset>_<subset>.csv`.

In [4]:
from preprocessing.filter_landmarks import filter_and_save

suffix        = '' if IMPUTE else '_no_imputation'
processed_csv = os.path.join(PROCESSED_DIR, DATASET, f'{DATASET}_{SUBSET}{suffix}.csv')

if os.path.exists(processed_csv):
    print(f'CSV já existe: {processed_csv}\nDelete o arquivo se quiser re-filtrar.')
else:
    filter_and_save(
        input_path   = interim_csv,
        output_path  = processed_csv,
        subset_name  = SUBSET,
        impute       = IMPUTE,
        dataset_name = DATASET,
    )
    print(f'Filtro concluído → {processed_csv}')

CSV já existe: data/processed/ksl/ksl_all.csv
Delete o arquivo se quiser re-filtrar.


In [5]:
# Carrega e inspeciona o DataFrame final
import pandas as pd
from datasets.base_dataset import BaseCsvDataset

if DATASET in CSV_DATASETS:
    loader = BaseCsvDataset.create(DATASET, INTERIM_DIR)
    df = loader.load()
else:
    df = pd.read_csv(processed_csv)

print(f'Shape:       {df.shape}')
print(f'Vídeos:      {df["video_name"].nunique()}')
print(f'Pessoas:     {sorted(df["person"].unique())}')
print(f'Categorias:  {df["category"].nunique()}')
df.head(3)

Shape:       (108373, 1636)
Vídeos:      1229
Pessoas:     [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Categorias:  67


,sign,category,person,video_name,frame,hand_0_0_x,hand_0_0_y,hand_0_0_z,hand_0_1_x,hand_0_1_y,...,pose_30_y,pose_30_z,pose_31_x,pose_31_y,pose_31_z,pose_32_x,pose_32_y,pose_32_z,missing_hand,missing_face
0,hi,0,0,00_01.MP4,0,0.579847,0.949595,3.003216e-08,0.548328,0.950875,...,1.807853,0.250683,0.592675,1.871811,0.192679,0.480322,1.859148,-0.035175,False,False
1,hi,0,0,00_01.MP4,1,0.584365,0.939701,7.236433e-09,0.556417,0.934509,...,1.806839,0.235043,0.592683,1.870546,0.187968,0.482592,1.860547,-0.023532,False,False
2,hi,0,0,00_01.MP4,2,0.587292,0.941082,1.461811e-08,0.562637,0.935167,...,1.815134,0.361336,0.588037,1.882730,0.219705,0.481584,1.876135,0.101787,False,False


---
## 3. Treino LOPO
### 3a. Fold único

In [6]:
import gc

# 1. Converte pra float32 (metade da memória)
landmark_cols = [c for c in df.columns if c.endswith(('_x', '_y', '_z'))]
df[landmark_cols] = df[landmark_cols].astype('float32')

# 2. Mantém só o necessário
keep = ['category', 'video_name', 'frame', 'person'] + landmark_cols
df = df[[c for c in keep if c in df.columns]].copy()

# 3. Força limpeza
gc.collect()

print(f'Memória do df: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')

Memória do df: 715.9 MB


In [13]:
from training.trainer import Trainer

# VALIDATE_PEOPLE = [1]
# TEST_PEOPLE     = [0]
VALIDATE_PEOPLE = [0, 1, 2, 3]
TEST_PEOPLE     = [0, 1, 2, 3]

result = Trainer({
    'dataset_name':    DATASET,
    'ref':             REF,
    'seed':            SEED,
    'validate_people': VALIDATE_PEOPLE,
    'test_people':     TEST_PEOPLE,
    'learning_rate':   LEARNING_RATE,
    'weight_decay':    WEIGHT_DECAY,
    'image_method':    IMAGE_METHOD,
    'model':           MODEL,
    'epochs':          EPOCHS,
    'output_dir':      OUTPUT_DIR,
    'augment_cfg': {
        'rotation_sigma':       12,
        'zoom_sigma':           0.1,
        'translate_x_sigma':    0.06,
        'horizontal_flip_prob': 0.3,
    },
}).run(df)

print(f"acc={result['test_accuracy']:.4f}  f1={result['test_f1']:.4f}")

/opt/homebrew/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/homebrew/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Epochs:   3%|▎         | 1/30 [00:28<13:45, 28.46s/it]

Epoch 1/30 | loss=4.2343 acc=0.0164 | val_loss=4.2495 val_acc=0.0078 | 2026-06-23 02:35:03.497626


Epochs:   7%|▋         | 2/30 [00:54<12:36, 27.03s/it]

Epoch 2/30 | loss=4.2206 acc=0.0185 | val_loss=4.4039 val_acc=0.0157 | 2026-06-23 02:35:29.528412


Epochs:  10%|█         | 3/30 [01:18<11:37, 25.84s/it]

...

Epochs:  17%|█▋        | 5/30 [02:07<10:18, 24.74s/it]

Epoch 5/30 | loss=4.2074 acc=0.0195 | val_loss=6.3709 val_acc=0.0157 | 2026-06-23 02:36:42.466167


Epochs:  17%|█▋        | 5/30 [02:28<12:24, 29.79s/it]

Early stopping at epoch 6.



Test — acc=0.0196 prec=0.0012 rec=0.0187 f1=0.0022
Results → experiments/ksl_all_com_imputacao/ksl/2026-06-23_02-37-06.json
acc=0.0196  f1=0.0022


In [14]:
print(f"Colunas _x: {len([c for c in df.columns if c.endswith('_x')])}")
print(f"Categorias: {df['category'].nunique()}")
print(f"Vídeos por person: {df.groupby('person')['video_name'].nunique()}")

Colunas _x: 543
Categorias: 67
Vídeos por person: person
0     66
1     61
2     61
3     67
4     56
5     46
6     37
7     65
8     63
9     62
10    67
11    64
12    64
13    65
14    62
15    65
16    64
17    65
18    64
19    65
Name: video_name, dtype: int64


In [15]:
print(f"Treino: {df[~df['person'].isin([0, 1])]['video_name'].nunique()} vídeos")
print(f"Val:    {df[df['person'] == 1]['video_name'].nunique()} vídeos")
print(f"Teste:  {df[df['person'] == 0]['video_name'].nunique()} vídeos")

Treino: 1102 vídeos
Val:    61 vídeos
Teste:  66 vídeos


### 3b. LOPO completo (todos os folds)

In [16]:
from training.trainer import Trainer

LOPO_FOLDS = {
    'minds':     [(i % 12 + 1, i + 1) for i in range(12)],
    'ufop':      [(1,2),(2,3),(3,4),(4,5),(5,1)],
    'ksl': [
        ([0,1,2,3],   [0,1,2,3]),
        ([4,5,6,7],   [4,5,6,7]),
        ([8,9,10,11], [8,9,10,11]),
        ([12,13,14,15],[12,13,14,15]),
        ([16,17,18,19],[16,17,18,19]),
    ],
    # 'ksl':       [(i % 20 + 1, i) for i in range(20)],
    'include50': [(i % 22 + 1, i) for i in range(22)],
}

BASE_CFG = {
    'dataset_name':  DATASET,
    'ref':           REF,
    'seed':          SEED,
    'learning_rate': LEARNING_RATE,
    'weight_decay':  WEIGHT_DECAY,
    'image_method':  IMAGE_METHOD,
    'model':         MODEL,
    'epochs':        EPOCHS,
    'output_dir':    OUTPUT_DIR,
    'augment_cfg': {
        'rotation_sigma':       12,
        'zoom_sigma':           0.1,
        'translate_x_sigma':    0.06,
        'horizontal_flip_prob': 0.5,
    },
}

folds   = LOPO_FOLDS[DATASET]
results = []

for i, (val, test) in enumerate(folds):
    val_list  = val  if isinstance(val,  list) else [val]
    test_list = test if isinstance(test, list) else [test]
    print(f'\nFold {i+1}/{len(folds)} — val={val_list} test={test_list}')

    r = Trainer({**BASE_CFG,
                 'validate_people': val_list,
                 'test_people':     test_list}).run(df)
    results.append(r)
    print(f'✓ acc={r["test_accuracy"]:.4f}  f1={r["test_f1"]:.4f}')

print(f'\n{len(folds)} folds concluídos.')


Fold 1/5 — val=[0, 1, 2, 3] test=[0, 1, 2, 3]


/opt/homebrew/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/homebrew/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Epochs:   3%|▎         | 1/30 [00:25<12:09, 25.17s/it]

Epoch 1/30 | loss=4.2343 acc=0.0164 | val_loss=4.2495 val_acc=0.0078 | 2026-06-23 02:37:52.843839


Epochs:   7%|▋         | 2/30 [00:50<11:41, 25.04s/it]

Epoch 2/30 | loss=4.2206 acc=0.0185 | val_loss=4.4039 val_acc=0.0157 | 2026-06-23 02:38:17.800022


...

Epochs:  17%|█▋        | 5/30 [02:07<10:53, 26.12s/it]

Epoch 5/30 | loss=4.2110 acc=0.0196 | val_loss=10.5464 val_acc=0.0116 | 2026-06-23 02:50:24.055252


Epochs:  17%|█▋        | 5/30 [02:32<12:41, 30.44s/it]

Early stopping at epoch 6.



Test — acc=0.0194 prec=0.0012 rec=0.0187 f1=0.0020
Results → experiments/ksl_all_com_imputacao/ksl/2026-06-23_02-50-51.json
✓ acc=0.0194  f1=0.0020

5 folds concluídos.


---
## 4. Resultados

In [17]:
import json, numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

results_path = os.path.join(OUTPUT_DIR, REF, DATASET)
loaded = [
    json.load(open(os.path.join(results_path, f)))
    for f in sorted(os.listdir(results_path))
    if f.endswith('.json')
]

print(f'{DATASET} #{REF} — {len(loaded)} folds')
print(f'model: {loaded[0].get("model")} | method: {loaded[0].get("image_method")}\n')

METRICS = [
    ('Accuracy',  accuracy_score,  {}),
    ('Precision', precision_score, {'average':'macro','zero_division':0}),
    ('Recall',    recall_score,    {'average':'macro','zero_division':0}),
    ('F1',        f1_score,        {'average':'macro','zero_division':0}),
]

summary = {}
for name, fn, kw in METRICS:
    vals = [fn(r['true_labels'], r['predicted_labels'], **kw) for r in loaded]
    summary[name] = {'mean': np.mean(vals), 'std': np.std(vals), 'folds': vals}
    print(f'  {name:<12} {np.mean(vals):.4f} ± {np.std(vals):.4f}')

ksl #ksl_all_com_imputacao — 6 folds
model: resnet18 | method: Skeleton-DML

  Accuracy     0.0210 ± 0.0021
  Precision    0.0013 ± 0.0006
  Recall       0.0203 ± 0.0025
  F1           0.0024 ± 0.0010


In [18]:
# Salva tabela CSV para dissertação
import pandas as pd

os.makedirs(TABLE_DIR, exist_ok=True)
table_path = os.path.join(TABLE_DIR, f'results_{DATASET}_{SUBSET}_ref{REF}.csv')

row = {'dataset': DATASET, 'subset': SUBSET, 'ref': REF,
       'model': loaded[0].get('model'), 'image_method': loaded[0].get('image_method'),
       'folds': len(loaded)}
for name, vals in summary.items():
    row[name]          = round(vals['mean'], 4)
    row[f'{name}_std'] = round(vals['std'],  4)

pd.DataFrame([row]).to_csv(table_path, index=False)
print(f'Tabela salva → {table_path}')

Tabela salva → reports/tables/results_ksl_all_refksl_all_com_imputacao.csv


---
## 5. Benchmark de latência

In [20]:
import cv2
from time import time
from tqdm.notebook import tqdm
from datasets.base_dataset import BaseVideoDataset
from extraction.base_extractor import BaseExtractor

BENCH_EXTRACTOR = 'mediapipe'
N_VIDEOS        = 5

if DATASET in CSV_DATASETS:
    print(f"Benchmark de latência não aplicável para '{DATASET}' (dataset CSV pré-processado).")
else:
    def video_frames(path):
        cap = cv2.VideoCapture(path)
        n   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()
        return n

    dataset_instance = BaseVideoDataset.create(DATASET, RAW_DIR)
    extractor        = BaseExtractor.create(BENCH_EXTRACTOR)
    videos           = dataset_instance.prepare_data()
    sample           = sorted(videos, key=lambda v: video_frames(v[0]))
    mid              = len(sample) // 2
    sample           = sample[:N_VIDEOS] + sample[mid:mid+N_VIDEOS] + sample[-N_VIDEOS:]

    total_time, total_frames = 0.0, 0
    for video_path, *_ in tqdm(sample):
        n  = video_frames(video_path)
        t0 = time()
        for _ in extractor.get_video_landmarks(video_path):
            pass
        total_time   += time() - t0
        total_frames += n

    print(f'Extractor:   {BENCH_EXTRACTOR}')
    print(f'FPS médio:   {total_frames / total_time:.2f}')
    print(f'Tempo/vídeo: {total_time / len(sample):.3f}s')

Benchmark de latência não aplicável para 'ksl' (dataset CSV pré-processado).
